# Day 42 — Section-aware chunking + local embeddings

Status: COMPLETE. 80 notes → 506 section-bounded, sentence-packed chunks
(metadata: doc/specialty/section); nomic-embed-text via Ollama → (506, 768)
L2-normalized. 8/8 tests pass.

In [1]:
import sys
sys.path.insert(0, "..")
import json
import numpy as np
from pathlib import Path

chunks = json.loads(Path("../data/chunks.json").read_text())
mat = np.load("../data/embeddings.npy")
lens = [len(c["text"]) for c in chunks]
print(f"notes: 80 -> chunks: {len(chunks)} "
      f"(avg {sum(lens)/len(lens):.0f} chars, max {max(lens)} = one long sentence alone)")
print(f"embeddings: {mat.shape} {mat.dtype}, unit norms "
      f"{'✓' if np.allclose(np.linalg.norm(mat, axis=1), 1, atol=1e-4) else '✗'} "
      "(inner product == cosine)")
print("example:", chunks[0]["text"][:80] + "...")

notes: 80 -> chunks: 506 (avg 540 chars, max 1678 = one long sentence alone)
embeddings: (506, 768) float32, unit norms ✓ (inner product == cosine)
example: [Cardiovascular / Pulmonary | HISTORY] 68-year-old male with ...


## Findings

1. **Sections preserved the facts**: FINDINGS vs IMPRESSION never mix — a
   retrieved chunk is a complete clinical statement, not a fragment.
2. **Every chunk self-describes** (`[specialty | section]` prefix): embeddings
   of bare sentences lack domain context; the prefix fixes that for free.
3. **Normalization is the cosine contract**: FAISS inner-product on unit vectors
   ranks identically to cosine — verified numerically, not assumed.